In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
!pip install -q -U langchain-core langchain-community langchain-google-genai langchain-huggingface faiss-cpu sentence-transformers google-generativeai pypdf

In [20]:
import os
from google.colab import drive, userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [21]:
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

VECTORSTORE_PATH = "/content/drive/MyDrive/Project_02/data/vectorstore/faiss"



print("--- Đang tải mô hình Embedding  ---")
embeddings = HuggingFaceEmbeddings(
    model_name="bkai-foundation-models/vietnamese-bi-encoder",
    model_kwargs={'device': 'cpu'}
)



--- Đang tải mô hình Embedding  ---


In [22]:
print("--- Đang nạp Vectorstore  ---")

db = FAISS.load_local(
    VECTORSTORE_PATH,
    embeddings,
    allow_dangerous_deserialization=True
)

# Lấy 3 đoạn văn bản có độ tương đồng cao nhất
retriever = db.as_retriever(search_kwargs={"k": 3})



--- Đang nạp Vectorstore  ---


In [23]:
print("--- Đang khởi tạo mô hình LLM ---")
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0.1,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)



--- Đang khởi tạo mô hình LLM ---


In [24]:
template = """
Bạn là một trợ lý luật sư AI chuyên nghiệp về Bảo mật, Dữ liệu và An ninh mạng.
Hãy sử dụng ngữ cảnh pháp lý dưới đây để trả lời câu hỏi của người dùng một cách chính xác nhất.

Ngữ cảnh pháp lý:
{context}

Câu hỏi: {question}

Yêu cầu trả lời:
1. Nếu thông tin không có trong văn bản, hãy nói "Mình xin lỗi, thông tin này không nằm trong cơ sở dữ liệu của mình".
2. Trích dẫn cụ thể Điều/Khoản nếu có trong văn bản.
3. Trình bày rõ ràng, dễ hiểu.

Trả lời:
"""

prompt = ChatPromptTemplate.from_template(template)



In [25]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)



In [27]:
question = "An ninh mạng là gì ?"

print(f"\nCâu hỏi: {question}")
try:
    response = rag_chain.invoke(question)
    print(f"\nTrả lời từ AI:\n{'-'*20}\n{response}")
except Exception as e:
    print(f"Có lỗi xảy ra: {e}")


Câu hỏi: An ninh mạng là gì ?

Trả lời từ AI:
--------------------
Theo quy định tại **Điều 2, Khoản 1 của LUẬT AN NINH MẠNG**, **An ninh mạng** được giải thích như sau:

An ninh mạng là sự bảo đảm hoạt động trên không gian mạng không gây phương hại đến an ninh quốc gia, trật tự, an toàn xã hội, quyền và lợi ích hợp pháp của cơ quan, tổ chức, cá nhân.
